# Baselines and Hyperparameter Selection



This notebook performs the hyperparameter search described in Section 3.5 and
establishes the untruncated reference performance of all three models. It runs
before the initialization and truncation studies because both of those depend on
its output: the configurations selected here are frozen and reused unchanged for
every subsequent condition.

The search follows the categories set out in Section 3.5.4. The truncation point
and the initialization method are independent variables and are *not* searched —
they are held at $k = T$ and Gaussian respectively, so that the nuisance
parameters are selected under a single fixed condition and can then be frozen.
Re-tuning them per condition would fold tuning effects into the very curves the
study is trying to read.

Two decisions from Section 3.5 shape what follows:

- **Substrate parameters are shared, optimisation parameters are not.** The leak
  rate and firing threshold define the substrate, so the LSM and the proposed
  model must adopt the same values or they would differ in two respects and the
  comparison between them would no longer isolate reservoir training. They are
  therefore searched on the LSM and inherited by the SNN — the conservative
  direction. The learning rate belongs to the optimisation problem rather than
  the substrate and is tuned per model.
- **The search is exhaustive.** A staged coordinate-descent alternative was
  measured on ECG5000 and missed the optimum by 0.0102 Macro-F1, its selection
  ranking tenth of ninety-six, so the separability it assumes does not hold here.

In [1]:
import pandas as pd

from edgesnn.data_utils import make_datasets
from edgesnn.experiments import (
    RESULTS_DIR, build_model, ensure_results_dir, load_config, run_grid, save_config,
)
from edgesnn.training import SELECTION_FOLDS
from edgesnn.tuning import BATCH_SIZES, DEFAULT_GRIDS, expand_grid, grid_search

DATA_DIR = "../Data"
ensure_results_dir()
SEED = 0

# InsectSound is by far the most expensive: one exhaustive SNN search there is on
# the order of half a day even on the GPU backend. Start with the two cheap
# datasets, confirm the pipeline, then add InsectSound to this list.
DATASETS = ["ECG5000", "Wafer"]

for kind, grid in DEFAULT_GRIDS.items():
    print(f"{kind:<5} {len(expand_grid(grid)):>3} configurations")
print("\nselection folds:", SELECTION_FOLDS)
print("batch sizes:    ", BATCH_SIZES)

snn    96 configurations
lsm    24 configurations
lstm    4 configurations

selection folds: {'ECG5000': 5, 'Wafer': 5, 'InsectSound': 1}
batch sizes:     {'ECG5000': 64, 'Wafer': 64, 'InsectSound': 256}


In [2]:
splits = {}
for name in DATASETS:
    train_ds, test_ds = make_datasets(DATA_DIR, name)
    splits[name] = (train_ds, test_ds)
    print(f"{name:<12} train={len(train_ds):>6}  test={len(test_ds):>6}  "
          f"T={train_ds.seq_len:>4}  C={train_ds.num_classes}")

ECG5000      train=   500  test=  4500  T= 140  C=2
Wafer        train=  1000  test=  6164  T= 152  C=2


## 1. Substrate search on the LSM

The neuron parameters are selected first, on the Liquid State Machine. Because
its reservoir is frozen, no gradient passes through a spike, and the search is
correspondingly cheap; more importantly, selecting them here and handing them to
the proposed model means no part of any later advantage can be attributed to a
substrate tuned in the proposed model's favour.

Selection is by Macro-F1 on a validation partition drawn from the training split,
cross-validated over five folds on ECG5000 and Wafer. The standard deviation
across folds is reported alongside the mean, because a difference between two
configurations is only meaningful when it exceeds the variation between folds.

In [3]:
substrate = {}

for name, (train_ds, _) in splits.items():
    cached = load_config(f"config_lsm_{name}")
    if cached is not None:
        substrate[name] = cached
        print(f"{name}: loaded cached configuration {cached}")
        continue

    table = grid_search(
        lambda **kw: build_model("lsm", num_classes=train_ds.num_classes,
                                 k=train_ds.seq_len, seed=SEED, **kw),
        train_ds, DEFAULT_GRIDS["lsm"], seed=SEED, max_epochs=60, min_epochs=15,
        patience=10, desc=f"LSM · {name}",
        train_kwargs={"batch_size": BATCH_SIZES[name]},
    )
    table.to_csv(f"{RESULTS_DIR}/search_lsm_{name}.csv", index=False)

    best = table.iloc[0]
    substrate[name] = {"leak_rate": float(best.leak_rate),
                       "threshold": float(best.threshold),
                       "lr": float(best.lr)}
    save_config(f"config_lsm_{name}", substrate[name])
    display(table.head(5))

ECG5000: loaded cached configuration {'leak_rate': 0.95, 'threshold': 0.5, 'lr': 0.003}
Wafer: loaded cached configuration {'leak_rate': 0.95, 'threshold': 0.5, 'lr': 0.003}


## 2. Learning rate and surrogate for the proposed model

The substrate settled, the proposed model is searched over its own learning rate
and surrogate family, with the leak rate and threshold pinned to the values
inherited from the LSM.

The two are searched jointly rather than in sequence. As Section 2.3.2 derives,
the four surrogate families return derivatives of $1$, $\gamma/2$, $1$ and
$\gamma/4$ at the threshold, so at $\gamma = 10$ their gradient magnitudes differ
by a factor of five; a family evaluated at a learning rate suited to another
would be judged on its gradient scale rather than on its shape.

In [4]:
snn_config = {}

for name, (train_ds, _) in splits.items():
    cached = load_config(f"config_snn_{name}")
    if cached is not None:
        snn_config[name] = cached
        print(f"{name}: loaded cached configuration {cached}")
        continue

    fixed = {k: substrate[name][k] for k in ("leak_rate", "threshold")}
    grid = {"lr": DEFAULT_GRIDS["snn"]["lr"],
            "surrogate": DEFAULT_GRIDS["snn"]["surrogate"]}

    table = grid_search(
        lambda **kw: build_model("snn", num_classes=train_ds.num_classes,
                                 k=train_ds.seq_len, seed=SEED, **fixed, **kw),
        train_ds, grid, seed=SEED, max_epochs=80, min_epochs=15, patience=12,
        desc=f"SNN · {name}", train_kwargs={"batch_size": BATCH_SIZES[name]},
    )
    table.to_csv(f"{RESULTS_DIR}/search_snn_{name}.csv", index=False)

    best = table.iloc[0]
    snn_config[name] = {**fixed, "surrogate": str(best.surrogate),
                        "lr": float(best.lr)}
    save_config(f"config_snn_{name}", snn_config[name])
    display(table.head(5))

ECG5000: loaded cached configuration {'leak_rate': 0.95, 'threshold': 0.5, 'surrogate': 'triangular', 'lr': 0.003}


SNN · Wafer:   0%|          | 0/16 [00:00<?, ?cfg/s]

  cfg 1/16 fold 1/5:   0%|          | 0/80 [00:00<?, ?ep/s]

  cfg 1/16 fold 2/5:   0%|          | 0/80 [00:00<?, ?ep/s]

  cfg 1/16 fold 3/5:   0%|          | 0/80 [00:00<?, ?ep/s]

...

  cfg 16/16 fold 2/5:   0%|          | 0/80 [00:00<?, ?ep/s]

  cfg 16/16 fold 3/5:   0%|          | 0/80 [00:00<?, ?ep/s]

  cfg 16/16 fold 4/5:   0%|          | 0/80 [00:00<?, ?ep/s]

  cfg 16/16 fold 5/5:   0%|          | 0/80 [00:00<?, ?ep/s]

,lr,surrogate,val_macro_f1,val_macro_f1_std,mean_best_epoch,n_splits
0,0.0030,sigmoid,0.982803,0.013896,20.0,5
1,0.0010,sigmoid,0.977261,0.011530,16.0,5
2,0.0003,sigmoid,0.974142,0.014539,18.2,5
3,0.0001,sigmoid,0.970911,0.015950,33.8,5
4,0.0010,triangular,0.966463,0.018052,17.4,5


The interaction between learning rate and surrogate family can be read directly
off the search table. If each family attains its best score at a different
learning rate, the two could not have been tuned independently without
misjudging at least one of them.

In [5]:
for name in splits:
    table = pd.read_csv(f"{RESULTS_DIR}/search_snn_{name}.csv")
    pivot = table.pivot_table(index="surrogate", columns="lr", values="val_macro_f1")
    print(f"\n{name}: validation Macro-F1 by surrogate family and learning rate")
    display(pivot.round(4))
    print("best learning rate per family:")
    display(table.loc[table.groupby("surrogate").val_macro_f1.idxmax()]
            [["surrogate", "lr", "val_macro_f1", "val_macro_f1_std"]]
            .reset_index(drop=True))


ECG5000: validation Macro-F1 by surrogate family and learning rate


lr,0.0001,0.0003,0.0010,0.0030
surrogate,,,,
arctan,0.9650,0.9712,0.9733,0.9733
fast_sigmoid,0.9610,0.9712,0.9774,0.9774
sigmoid,0.9631,0.9630,0.9774,0.9794
triangular,0.9610,0.9753,0.9732,0.9814


best learning rate per family:


,surrogate,lr,val_macro_f1,val_macro_f1_std
0,arctan,0.001,0.973292,0.021132
1,fast_sigmoid,0.001,0.977394,0.013714
2,sigmoid,0.003,0.979435,0.014475
3,triangular,0.003,0.981432,0.010157



Wafer: validation Macro-F1 by surrogate family and learning rate


lr,0.0001,0.0003,0.0010,0.0030
surrogate,,,,
arctan,0.7755,0.8507,0.9590,0.9598
fast_sigmoid,0.9521,0.9565,0.9626,0.9588
sigmoid,0.9709,0.9741,0.9773,0.9828
triangular,0.9432,0.9604,0.9665,0.9641


best learning rate per family:


,surrogate,lr,val_macro_f1,val_macro_f1_std
0,arctan,0.003,0.959827,0.038247
1,fast_sigmoid,0.001,0.962590,0.017988
2,sigmoid,0.003,0.982803,0.013896
3,triangular,0.001,0.966463,0.018052


## 3. Learning rate for the LSTM

The dense baseline has no neuron or surrogate parameters, so only its learning
rate is searched. Tuning it separately rather than imposing the spiking model's
rate is the fair choice: a single shared learning rate would penalise whichever
architecture happened to dislike it, and the resulting difference would be an
artefact of that choice rather than a property of the models.

In [6]:
lstm_config = {}

for name, (train_ds, _) in splits.items():
    cached = load_config(f"config_lstm_{name}")
    if cached is not None:
        lstm_config[name] = cached
        print(f"{name}: loaded cached configuration {cached}")
        continue

    table = grid_search(
        lambda **kw: build_model("lstm", num_classes=train_ds.num_classes,
                                 k=train_ds.seq_len, seed=SEED, **kw),
        train_ds, DEFAULT_GRIDS["lstm"], seed=SEED, max_epochs=60, min_epochs=15,
        patience=10, desc=f"LSTM · {name}",
        train_kwargs={"batch_size": BATCH_SIZES[name]},
    )
    table.to_csv(f"{RESULTS_DIR}/search_lstm_{name}.csv", index=False)

    lstm_config[name] = {"lr": float(table.iloc[0].lr)}
    save_config(f"config_lstm_{name}", lstm_config[name])
    display(table)

LSTM · ECG5000:   0%|          | 0/4 [00:00<?, ?cfg/s]

  cfg 1/4 fold 1/5:   0%|          | 0/60 [00:00<?, ?ep/s]

  cfg 1/4 fold 2/5:   0%|          | 0/60 [00:00<?, ?ep/s]

  cfg 1/4 fold 3/5:   0%|          | 0/60 [00:00<?, ?ep/s]

  cfg 1/4 fold 4/5:   0%|          | 0/60 [00:00<?, ?ep/s]

...

  cfg 4/4 fold 2/5:   0%|          | 0/60 [00:00<?, ?ep/s]

  cfg 4/4 fold 3/5:   0%|          | 0/60 [00:00<?, ?ep/s]

  cfg 4/4 fold 4/5:   0%|          | 0/60 [00:00<?, ?ep/s]

  cfg 4/4 fold 5/5:   0%|          | 0/60 [00:00<?, ?ep/s]

,lr,val_macro_f1,val_macro_f1_std,mean_best_epoch,n_splits
0,0.0030,0.679677,0.164961,14.4,5
1,0.0010,0.667338,0.138296,13.0,5
2,0.0003,0.592773,0.133197,8.0,5
3,0.0001,0.484763,0.020786,1.8,5


## 4. Untruncated reference performance

With the configurations fixed, each model is trained at $k = T$ under the
Gaussian initialization and evaluated on the test split. This is the only point
at which the test split is touched, and these figures are the reference against
which every truncated result in the following notebooks is read: they state what
each architecture achieves when it is given the entire sequence.

The stochastic conditions are repeated over five seeds so that later differences
can be judged against seed-to-seed variation.

In [7]:
configs = {"snn": snn_config, "lsm": substrate, "lstm": lstm_config}
reference = {}

for name, (train_ds, test_ds) in splits.items():
    conditions = [
        dict(kind=kind, k=train_ds.seq_len, init_method="gaussian", seed=seed,
             config=dict(configs[kind][name]))
        for kind in ("snn", "lsm", "lstm")
        for seed in range(5)
    ]
    reference[name] = run_grid(f"baselines_{name}", conditions, train_ds, test_ds,
                               max_epochs=80, min_epochs=15, patience=12)

baselines_ECG5000: 15 conditions to run


baselines_ECG5000:   0%|          | 0/15 [00:00<?, ?run/s]

baselines_Wafer: 15 conditions to run


baselines_Wafer:   0%|          | 0/15 [00:00<?, ?run/s]

In [8]:
summary = []
for name, frame in reference.items():
    grouped = frame.groupby("model").agg(
        macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
        accuracy=("accuracy", "mean"), majority=("majority_rate", "mean"),
        macs=("macs", "mean"), sops=("sops", "mean"), energy_pj=("energy_pj", "mean"),
    ).reset_index()
    grouped.insert(0, "dataset", name)
    summary.append(grouped)

summary = pd.concat(summary, ignore_index=True)
summary["energy_uJ"] = summary.energy_pj / 1e6
display(summary.round(4))

,dataset,model,macro_f1,macro_f1_sd,accuracy,majority,macs,sops,energy_pj,energy_uJ
0,ECG5000,lsm,0.9433,0.0179,0.9456,0.5838,36352.0,332023.2500,4.660401e+05,0.4660
1,ECG5000,lstm,0.9647,0.0165,0.9661,0.5838,9246976.0,0.0000,4.253609e+07,42.5361
2,ECG5000,snn,0.9557,0.0064,0.9572,0.5838,36352.0,291829.5562,4.298658e+05,0.4299
3,Wafer,lsm,0.9027,0.0065,0.9645,0.8921,39424.0,369196.9312,5.136276e+05,0.5136
4,Wafer,lstm,0.7263,0.0890,0.8567,0.8921,10039552.0,0.0000,4.618194e+07,46.1819
5,Wafer,snn,0.9325,0.0101,0.9748,0.8921,39424.0,334578.0562,4.824707e+05,0.4825


## 5. What this establishes

Three quantities carry forward. The selected configurations are frozen from this
point and are not revisited: every condition in the initialization and truncation
studies uses them unchanged, which is what allows those results to be attributed
to the initializer and the cutoff rather than to tuning.

The untruncated Macro-F1 of each model gives the ceiling against which its
truncated variants are compared, and the majority-class rate reported beside it
gives the floor. A model whose score sits near that floor has not learned the
minority class at all, which on Wafer is the only class of practical interest.

Finally, the operation counts at $k = T$ establish the untruncated energy of each
architecture. Since both the dense and spiking costs scale with the number of
time steps executed, these figures set the scale against which the savings from
truncation are measured in Notebook 05.